# 스트레스 점수 예측 V23 — Categorical Preprocessing Ensemble

- 검증된 V7 Pair-Neighbor를 기준으로 고정
- 범주형 결측 플래그, 범주 상호작용, Fold Train 빈도 인코딩 추가
- 기존 Tree와 개선 Tree를 혼합하고 Pair-Neighbor 15%는 유지
- 세 5-Fold 분할 모두에서 기준을 이기고 평균 0.0005 이상 개선될 때만 선택
- 빈도표·결측치·One-Hot은 각 Fold Train으로만 학습

In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

## 1. 설정과 데이터

In [2]:
DATA_DIR=Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR=Path("/Users/bitsaem/Desktop/stress_project")
CV_SEEDS=(42,77,2026)
ENHANCED_QUANTILES=(0.50,0.52,0.54)
ENHANCED_TREE_WEIGHTS=(0.25,0.50,0.75,1.00)

INTERACTIONS=[
 ("medical_history","family_medical_history"),
 ("activity","sleep_pattern"),
 ("smoke_status","activity"),
 ("smoke_status","sleep_pattern"),
 ("gender","activity"),
 ("edu_level","activity"),
]

train=pd.read_csv(DATA_DIR/"train.csv")
test=pd.read_csv(DATA_DIR/"test.csv")
submission=pd.read_csv(DATA_DIR/"sample_submission.csv")
validate_inputs(train,test,submission)
raw=train.drop(columns=[TARGET,ID_COLUMN]).reset_index(drop=True)
raw_test=test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
X=add_features(raw);X_test=add_features(raw_test)
y=train[TARGET].reset_index(drop=True)
print("Train:",X.shape,"| Test:",X_test.shape)

Train: (3000, 22) | Test: (3000, 22)


## 2. Train-only 범주형 강화 전처리

In [3]:
def categorical_enhancement_fit_transform(train_frame,valid_frame):
    train_out=train_frame.copy();valid_out=valid_frame.copy()
    categorical=train_frame.select_dtypes(
        include=["object","category","string"]
    ).columns.tolist()

    # 행 단위 결측 플래그와 범주 상호작용
    for column in categorical:
        train_out[f"{column}__missing"]=train_frame[column].isna().astype("int8")
        valid_out[f"{column}__missing"]=valid_frame[column].isna().astype("int8")

    interaction_columns=[]
    for first,second in INTERACTIONS:
        name=f"{first}__X__{second}"
        train_out[name]=(
            train_frame[first].fillna("__MISSING__").astype(str)+"|"+
            train_frame[second].fillna("__MISSING__").astype(str)
        )
        valid_out[name]=(
            valid_frame[first].fillna("__MISSING__").astype(str)+"|"+
            valid_frame[second].fillna("__MISSING__").astype(str)
        )
        interaction_columns.append(name)

    # 빈도표는 Train Fold로만 생성
    for column in categorical+interaction_columns:
        train_values=train_out[column].fillna("__MISSING__").astype(str)
        valid_values=valid_out[column].fillna("__MISSING__").astype(str)
        frequency=train_values.value_counts(normalize=True)
        train_out[f"{column}__frequency"]=train_values.map(frequency).astype(float)
        valid_out[f"{column}__frequency"]=valid_values.map(frequency).fillna(0).astype(float)
    return train_out,valid_out


def tree_prediction_material(train_frame,valid_frame,target,enhanced=False):
    if enhanced:
        train_frame,valid_frame=categorical_enhancement_fit_transform(
            train_frame,valid_frame
        )
    weighted_train=add_feature_copies(train_frame)
    weighted_valid=add_feature_copies(valid_frame)
    pipeline=Pipeline([
        ("preprocessor",make_preprocessor(weighted_train)),
        ("model",ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,min_samples_leaf=1,max_features=1,
            random_state=RANDOM_SEED,n_jobs=-1,
        )),
    ])
    pipeline.fit(weighted_train,target)
    matrix=pipeline.named_steps["preprocessor"].transform(weighted_valid)
    forest=pipeline.named_steps["model"]
    return np.column_stack([tree.predict(matrix) for tree in forest.estimators_])


def rounded_tree_quantile(material,quantile):
    return np.clip(np.round(np.quantile(material,quantile,axis=1),2),0,1)

## 3. 동일한 3개 CV 분할에서 OOF 생성

In [4]:
split_data={}
for cv_seed in CV_SEEDS:
    cv=KFold(5,shuffle=True,random_state=cv_seed)
    base_tree=np.zeros(len(y));pair=np.zeros(len(y))
    enhanced={q:np.zeros(len(y)) for q in ENHANCED_QUANTILES}
    for fold,(ti,vi) in enumerate(cv.split(X),1):
        base_material=tree_prediction_material(
            X.iloc[ti],X.iloc[vi],y.iloc[ti],enhanced=False
        )
        enhanced_material=tree_prediction_material(
            X.iloc[ti],X.iloc[vi],y.iloc[ti],enhanced=True
        )
        base_tree[vi]=rounded_tree_quantile(base_material,.52)
        for quantile in ENHANCED_QUANTILES:
            enhanced[quantile][vi]=rounded_tree_quantile(
                enhanced_material,quantile
            )
        pair[vi]=pair_neighbor_prediction(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        print(f"CV {cv_seed} | Fold {fold}: complete")
    baseline=np.clip(np.round(.85*base_tree+.15*pair,2),0,1)
    split_data[cv_seed]={
        "base_tree":base_tree,"enhanced":enhanced,"pair":pair,"baseline":baseline
    }
    print(f"CV {cv_seed} V7 MAE: {mean_absolute_error(y,baseline):.6f}")

CV 42 | Fold 1: complete
CV 42 | Fold 2: complete
CV 42 | Fold 3: complete
CV 42 | Fold 4: complete
CV 42 | Fold 5: complete
CV 42 V7 MAE: 0.148550
CV 77 | Fold 1: complete
CV 77 | Fold 2: complete
CV 77 | Fold 3: complete
CV 77 | Fold 4: complete
CV 77 | Fold 5: complete
CV 77 V7 MAE: 0.146387
CV 2026 | Fold 1: complete
CV 2026 | Fold 2: complete
CV 2026 | Fold 3: complete
CV 2026 | Fold 4: complete
CV 2026 | Fold 5: complete
CV 2026 V7 MAE: 0.144733


## 4. 강화 Tree 혼합 비율 선택

In [5]:
rows=[]
for quantile in ENHANCED_QUANTILES:
 for enhanced_weight in ENHANCED_TREE_WEIGHTS:
    scores=[];improvements=[]
    for cv_seed in CV_SEEDS:
        item=split_data[cv_seed]
        tree_blend=(
            (1-enhanced_weight)*item["base_tree"]+
            enhanced_weight*item["enhanced"][quantile]
        )
        prediction=np.clip(np.round(.85*tree_blend+.15*item["pair"],2),0,1)
        base_mae=mean_absolute_error(y,item["baseline"])
        score=mean_absolute_error(y,prediction)
        scores.append(score);improvements.append(base_mae-score)
    rows.append({
        "enhanced_quantile":quantile,"enhanced_tree_weight":enhanced_weight,
        "mean_CV_MAE":np.mean(scores),"mean_improvement":np.mean(improvements),
        "worst_improvement":min(improvements),"wins":sum(v>0 for v in improvements),
    })
results=pd.DataFrame(rows).sort_values(
    ["wins","worst_improvement","mean_CV_MAE"],ascending=[False,False,True]
).reset_index(drop=True)
eligible=results[(results.wins==3)&(results.worst_improvement>0)&(results.mean_improvement>=.0005)]
if len(eligible):selected=eligible.iloc[0]
else:selected=pd.Series({
 "enhanced_quantile":.52,"enhanced_tree_weight":0.0,
 "mean_CV_MAE":np.mean([mean_absolute_error(y,split_data[s]["baseline"]) for s in CV_SEEDS]),
 "mean_improvement":0.0,"worst_improvement":0.0,"wins":0,
})
print("Selected enhanced quantile:",selected.enhanced_quantile)
print("Selected enhanced tree weight:",selected.enhanced_tree_weight)
print("Repeated-CV mean MAE:",f"{selected.mean_CV_MAE:.6f}")
print("Mean improvement:",f"{selected.mean_improvement:+.6f}")
print("Worst split improvement:",f"{selected.worst_improvement:+.6f}")
print("Wins:",int(selected.wins),"/ 3")
results

Selected enhanced quantile: 0.52
Selected enhanced tree weight: 0.0
Repeated-CV mean MAE: 0.146557
Mean improvement: +0.000000
Worst split improvement: +0.000000
Wins: 0 / 3


,enhanced_quantile,enhanced_tree_weight,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,0.54,0.25,0.147030,-0.000473,-0.000577,0
1,0.50,0.25,0.147017,-0.000460,-0.000643,0
2,0.52,0.25,0.147019,-0.000462,-0.000733,0
3,0.54,0.50,0.147887,-0.001330,-0.001563,0
4,0.50,0.50,0.147844,-0.001288,-0.001573,0
5,0.52,0.50,0.147826,-0.001269,-0.001660,0
6,0.54,0.75,0.148947,-0.002390,-0.002670,0
7,0.52,0.75,0.148863,-0.002307,-0.002793,0
8,0.50,0.75,0.148959,-0.002402,-0.002803,0
9,0.54,1.00,0.150401,-0.003844,-0.004237,0


## 5. 전체 Train 학습과 Test 예

In [6]:
base_material=tree_prediction_material(X,X_test,y,enhanced=False)
base_tree=rounded_tree_quantile(base_material,.52)
enhanced_weight=float(selected.enhanced_tree_weight)
if enhanced_weight>0:
    enhanced_material=tree_prediction_material(X,X_test,y,enhanced=True)
    enhanced_tree=rounded_tree_quantile(
        enhanced_material,float(selected.enhanced_quantile)
    )
else:
    enhanced_tree=base_tree.copy()
tree_blend=(1-enhanced_weight)*base_tree+enhanced_weight*enhanced_tree
pair_prediction=pair_neighbor_prediction(X,X_test,y)
pred_v23=np.clip(np.round(.85*tree_blend+.15*pair_prediction,2),0,1)
print("Prediction range:",pred_v23.min(),pred_v23.max())
print("첫 10개:",pred_v23[:10])

Prediction range: 0.0 1.0
첫 10개: [0.56 0.97 0.19 0.48 0.53 0.47 0.78 0.48 0.57 0.88]


## 6. 제출 파일

In [7]:
submission_v23=submission.copy();submission_v23[TARGET]=pred_v23
output_path=OUTPUT_DIR/"submit_v23_categorical_preprocessing.csv"
submission_v23.to_csv(output_path,index=False,encoding="utf-8")
print("Saved:",output_path)
submission_v23.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v23_categorical_preprocessing.csv


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88


## 제출 기준

- `Wins: 3 / 3`
- `Mean improvement >= 0.0005`
- `Worst split improvement > 0`
- `Selected enhanced tree weight > 0`
- 조건 미달이면 Public 0.1272333 원본 V7 유지